# Amazon Pipeline - Full Test

Pipeline:
1. Search Amazon (curl_cffi) -> parse product cards
2. Filter on-sale + Scrape features (Approach A: specs, Approach B: product page)
3. Select top 5 deals (Cerebras via LiteLLM)
4. Estimate prices (EnsembleAgent)
5. Final results

In [1]:
# Cell 1: Setup
import os, sys, logging, time

os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, 'test_chuc_nang/fix_amazon_v2_s2')

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print(f'Working directory: {os.getcwd()}')
print(f'OPENROUTER_API_KEY: {"SET" if os.getenv("OPENROUTER_API_KEY") else "MISSING"}')
print('Setup complete!')

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
OPENROUTER_API_KEY: SET
Setup complete!


In [2]:
# Cell 2: Imports
import chromadb
from litellm import completion
from curl_cffi import requests as curl_requests

from price_agents.deals import Deal, DealSelection, Opportunity
from price_agents.ensemble_agent import EnsembleAgent
from buoc1_search import (
    ScrapedAmazonDeal,
    init_amazon_session,
    search_amazon,
    scrape_product_page,
    search_filter_scrape_amazon,
)
from bestbuy_untils.unified_deal import UnifiedScrapedDeal

print('Imports done!')

INFO:datasets:PyTorch version 2.9.0 available.


Imports done!


In [3]:
# Cell 3: Init EnsembleAgent (run once)
print('Initializing EnsembleAgent...')
client = chromadb.PersistentClient(path='products_vectorstore')
collection = client.get_or_create_collection('products')
print(f'ChromaDB: {collection.count()} documents')

ensemble = EnsembleAgent(collection)
print('EnsembleAgent ready!')

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing EnsembleAgent...
ChromaDB: 800000 documents


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


EnsembleAgent ready!


In [4]:
# Cell 4: Step 1+2 - Search + Filter + Scrape Amazon
TEST_KEYWORD = 'laptop gaming'

print('=' * 60)
print(f'STEP 1+2: Search + Filter + Scrape Amazon for "{TEST_KEYWORD}"')
print('=' * 60)

start = time.time()
scraped_deals = search_filter_scrape_amazon(TEST_KEYWORD, max_results=5)
elapsed = time.time() - start

print(f'\n{"="*60}')
print(f'Got {len(scraped_deals)} deals in {elapsed:.1f}s:')
for i, d in enumerate(scraped_deals, 1):
    print(f'  [{i}] ${d.price:.2f} | {d.title[:70]}')
    print(f'       Brand: {d.brand or "N/A"} | Features: {len(d.features)} chars')
    print(f'       URL: {d.url}')

STEP 1+2: Search + Filter + Scrape Amazon for "laptop gaming"


INFO:buoc1_search:[Init] ZIP 96150 set OK
INFO:buoc1_search:[Step 1] Search: https://www.amazon.com/s?k=laptop+gaming
INFO:buoc1_search:  Status: 200 | Size: 1,992,097 bytes
INFO:buoc1_search:  Found 22 products (12 on sale)
INFO:buoc1_search:[Step 2] Filtered: 12 on sale (from 22 total)
INFO:buoc1_search:
[Result] 5 deals in 14.1s (Approach A: 5, Approach B: 0)



Got 5 deals in 14.1s:
  [1] $299.98 | NIMO 15.6'' IPS FHD-Laptop, 8GB RAM 256GB SSD AMD Ryzen 5(Beat i5-1135
       Brand: N/A | Features: 85 chars
       URL: https://www.amazon.com/dp/B0DZ5LJXL3
  [2] $799.97 | NIMO 17.3 Gaming-Laptop Ryzen 9 8945HS (Beat i9-13900H, Up to 5.2GHz) 
       Brand: N/A | Features: 86 chars
       URL: https://www.amazon.com/dp/B0G516T6MJ
  [3] $749.99 | Dell Inspiron 15.6" FHD Touchscreen Laptop, 8-Core AMD Ryzen 7 7730U (
       Brand: N/A | Features: 84 chars
       URL: https://www.amazon.com/dp/B0GQYXYZ85
  [4] $549.99 | NIMO 15.6" IPS FHD Gaming-Laptop, 8 Cores AMD Ryzen 7 Pro 6850U 16GB L
       Brand: N/A | Features: 90 chars
       URL: https://www.amazon.com/dp/B0GN2B8MZP
  [5] $2299.00 | ASUS ROG Strix G18 (2025) Gaming Laptop, 18” ROG Nebula 16:10 2.5K 240
       Brand: N/A | Features: 89 chars
       URL: https://www.amazon.com/dp/B0F1CB49YB


In [5]:
# Cell 5: Convert to UnifiedScrapedDeal (same format as BestBuy)
unified_deals = [UnifiedScrapedDeal.from_amazon(d) for d in scraped_deals]

print(f'Converted {len(unified_deals)} deals to UnifiedScrapedDeal:')
for i, ud in enumerate(unified_deals, 1):
    print(f'\n[{i}] {ud.source}: {ud.title[:70]}')
    print(f'    ${ud.price:.2f} | {ud.brand or "N/A"}')
    print(f'    Features: {ud.features[:100]}...')

Converted 5 deals to UnifiedScrapedDeal:

[1] Amazon: NIMO 15.6'' IPS FHD-Laptop, 8GB RAM 256GB SSD AMD Ryzen 5(Beat i5-1135
    $299.98 | N/A
    Features: Display Size: 15.6 inches, Disk Size: 256 GB, RAM: 8 GB, Operating System: Windows 11...

[2] Amazon: NIMO 17.3 Gaming-Laptop Ryzen 9 8945HS (Beat i9-13900H, Up to 5.2GHz) 
    $799.97 | N/A
    Features: Display Size: 17.3 inches, Disk Size: 512 GB, RAM: 16 GB, Operating System: Windows 11...

[3] Amazon: Dell Inspiron 15.6" FHD Touchscreen Laptop, 8-Core AMD Ryzen 7 7730U (
    $749.99 | N/A
    Features: Display Size: 15.6 inches, Disk Size: 1 TB, RAM: 32 GB, Operating System: Windows 11...

[4] Amazon: NIMO 15.6" IPS FHD Gaming-Laptop, 8 Cores AMD Ryzen 7 Pro 6850U 16GB L
    $549.99 | N/A
    Features: Display Size: 15.60 inches, Disk Size: 512 GB, RAM: 16.00 GB, Operating System: Windows 11...

[5] Amazon: ASUS ROG Strix G18 (2025) Gaming Laptop, 18” ROG Nebula 16:10 2.5K 240
    $2299.00 | N/A
    Features: Display Size: 18 

In [6]:
from openai import OpenAI                                                                                                                                      
                                                                                                                                                                 
print('=' * 60)                                                                                                                                                
print('STEP 3: Select top 3 deals (GPT-5-mini)')                                                                                                               
print('=' * 60)                                                                                                                                                
                                                                                                                                                                
SYSTEM_PROMPT = """You identify and summarize the 3 most detailed deals from a list, by selecting deals that have the most detailed, high quality description  
and the most clear price.                                                                                                                                      
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description.                        
Most important is that you respond with the 3 deals that have the most detailed product description with price.                                                
                                                                                                                                                                
**IMPORTANT:**                                                                                                                                                 
1. Focus on the product features and specifications, not sales terms.                                                                                          
2. The product_description should be a 3-4 sentence summary of the product itself.                                                                             
3. Price must be greater than 0.                                                                                                                               
4. Keep the original URL exactly as provided."""                                                                                                               
                                                                                                                                                                
USER_PROMPT_PREFIX = """Respond with the most promising 3 deals from this list, selecting those which have the most detailed, high quality product description 
and a clear price that is greater than 0.                                                                                                                      
You should rephrase the description to be a summary of the product itself, not the terms of the deal.                                                          
Remember to respond with a short paragraph of text in the product_description field for each of the 3 items that you select.                                   
                                                                                                                                                                
Deals:                                                                                                                                                         
                                                                                                                                                                
"""                                                                                                                                                         

valid_deals = [ud for ud in unified_deals if ud.price > 0]
user_prompt = USER_PROMPT_PREFIX                                                                                                                               
user_prompt += '\n\n'.join([ud.describe() for ud in valid_deals])                                                                                              
user_prompt += '\n\nInclude up to 5 deals, no more.'                                                                                                           
                                                                                                                                                                
start = time.time()                                                                                                                                            
print(f'Calling GPT-5-mini with {len(valid_deals)} deals...')                                                                                                  
                                                                                                                                                            
openai_client = OpenAI()
result = openai_client.chat.completions.parse(                                                                                                                 
    model="gpt-5-mini",                                                                                                                                        
    messages=[                                                                                                                                                 
        {"role": "system", "content": SYSTEM_PROMPT},                                                                                                          
        {"role": "user", "content": user_prompt},                                                                                                           
    ],                                                                                                                                                         
    response_format=DealSelection,                                                                                                                             
    reasoning_effort="minimal",                                                                                                                                
)                                                                                                                                                              
                                                                                                                                                            
deal_selection = result.choices[0].message.parsed
deal_selection.deals = [d for d in deal_selection.deals if d.price > 0]                                                                                        
                                                                                                                                                                
elapsed = time.time() - start                                                                                                                                  
print(f'\nSelected {len(deal_selection.deals)} deals in {elapsed:.1f}s:')                                                                                      
for i, deal in enumerate(deal_selection.deals, 1):                                                                                                             
    print(f'  [{i}] ${deal.price:.2f} | {deal.product_description[:80]}...')                                                                                   
    print(f'       URL: {deal.url}')

STEP 3: Select top 3 deals (GPT-5-mini)
Calling GPT-5-mini with 5 deals...

Selected 3 deals in 10.3s:
  [1] $299.98 | 15.6-inch IPS Full HD laptop powered by an AMD Ryzen 5 processor (4 cores, boost...
       URL: https://www.amazon.com/dp/B0DZ5LJXL3
  [2] $799.97 | 17.3-inch gaming-oriented laptop featuring an AMD Ryzen 9 8945HS processor with ...
       URL: https://www.amazon.com/dp/B0G516T6MJ
  [3] $2299.00 | ASUS ROG Strix G18 (2025) is a high-end 18-inch gaming laptop with a 2.5K 16:10 ...
       URL: https://www.amazon.com/dp/B0F1CB49YB


In [7]:
# Cell 7: Step 4 - Estimate prices (EnsembleAgent)
print('=' * 60)
print('STEP 4: Estimate prices with EnsembleAgent')
print('=' * 60)

start = time.time()
opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f'\n[{i}/{len(deal_selection.deals)}] {deal.product_description[:50]}...')
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    opportunities.append(Opportunity(deal=deal, estimate=estimate, discount=discount))
    print(f'  Sale: ${deal.price:.2f} | Est: ${estimate:.2f} | Discount: ${discount:.2f}')

print(f'\nEstimation done in {time.time() - start:.1f}s')

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
11:49:31 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


STEP 4: Estimate prices with EnsembleAgent

[1/3] 15.6-inch IPS Full HD laptop powered by an AMD Ryz...


11:49:32 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $350.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $389.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $532.76
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $399.48
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
11:50:29 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

  Sale: $799.97 | Est: $1058.63 | Discount: $258.66

[3/3] ASUS ROG Strix G18 (2025) is a high-end 18-inch ga...


11:50:33 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $999.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $2499.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $1019.79
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $2201.08


  Sale: $2299.00 | Est: $2201.08 | Discount: $-97.92

Estimation done in 65.0s


In [8]:
# Cell 8: Final Results
print('=' * 80)
print('FINAL RESULTS - Amazon Deals Sorted by Discount')
print('=' * 80)
print(f'Keyword: "{TEST_KEYWORD}"\n')

opportunities.sort(key=lambda x: x.discount, reverse=True)

for i, opp in enumerate(opportunities, 1):
    pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    status = 'HOT DEAL' if opp.discount > 200 else 'Good Deal' if opp.discount > 100 else 'OK' if opp.discount > 0 else 'Overpriced'
    
    print(f'--- #{i} [{status}] ---')
    print(f'  Product:  {opp.deal.product_description[:80]}...')
    print(f'  Sale:     ${opp.deal.price:.2f}')
    print(f'  Estimate: ${opp.estimate:.2f}')
    print(f'  Discount: ${opp.discount:.2f} ({pct:.0f}%)')
    print(f'  URL:      {opp.deal.url}')
    print()

FINAL RESULTS - Amazon Deals Sorted by Discount
Keyword: "laptop gaming"

--- #1 [HOT DEAL] ---
  Product:  17.3-inch gaming-oriented laptop featuring an AMD Ryzen 9 8945HS processor with ...
  Sale:     $799.97
  Estimate: $1058.63
  Discount: $258.66 (24%)
  URL:      https://www.amazon.com/dp/B0G516T6MJ

--- #2 [OK] ---
  Product:  15.6-inch IPS Full HD laptop powered by an AMD Ryzen 5 processor (4 cores, boost...
  Sale:     $299.98
  Estimate: $399.48
  Discount: $99.50 (25%)
  URL:      https://www.amazon.com/dp/B0DZ5LJXL3

--- #3 [Overpriced] ---
  Product:  ASUS ROG Strix G18 (2025) is a high-end 18-inch gaming laptop with a 2.5K 16:10 ...
  Sale:     $2299.00
  Estimate: $2201.08
  Discount: $-97.92 (-4%)
  URL:      https://www.amazon.com/dp/B0F1CB49YB

